In [1]:
%load_ext autoreload
%autoreload 2

In [11]:
from climada.entity.measures import Measure, CostIncome, MeasureSet
from climada.hazard import Hazard
from climada.util import HAZ_DEMO_H5
from climada.entity import Entity
from climada.util.constants import ENT_DEMO_FUTURE, ENT_DEMO_TODAY
from climada.trajectories.snapshot import Snapshot
from climada.entity import ImpactFuncSet, ImpfTropCyclone
from climada.engine.option_appraisal.appraiser import StaticAppraiser
from climada.entity.measures.types import MeasureKwargs
import warnings

warnings.simplefilter("ignore")

# Load exposure
exp_present = Entity.from_excel(ENT_DEMO_TODAY).exposures
exp_future = Entity.from_excel(ENT_DEMO_FUTURE).exposures

# Load Hazard
haz_present = Hazard.from_hdf5(HAZ_DEMO_H5)
haz_future = Hazard.from_hdf5(HAZ_DEMO_H5)
haz_future.frequency *= 1.2

# Load impact function set
impf_tc_present = ImpfTropCyclone.from_emanuel_usa()
impf_tc_future = ImpfTropCyclone.from_emanuel_usa(v_half=70.0)
impf_set_present = ImpactFuncSet([impf_tc_present])
impf_set_future = ImpactFuncSet([impf_tc_future])

# Create the snapshots
snap1 = Snapshot(
    exposure=exp_present, hazard=haz_present, impfset=impf_set_present, date="2018"
)

snap2 = Snapshot(
    exposure=exp_future, hazard=haz_future, impfset=impf_set_future, date="2050"
)


# Define the effect with python functions
def targeted_campaign(impact_func_set, impf_id, rescale):
    """Rescales the impact function with id impf_id"""
    impact_func_set.get_func(fun_id=impf_id)[0].mdd *= rescale
    return impact_func_set


def building_retrofit(exposures, reduction_factor):
    """Reduces the value of all assets by a fixed percentage."""
    exposures.gdf["value"] *= 1 - reduction_factor
    return exposures


def sea_wall_protection(hazard, reduction):
    """Reduces flood intensity by an absolute amount."""
    hazard.intensity.data -= reduction
    return hazard


# Initialize the Measure objects
campaign = Measure(
    name="Targeted awareness campaign",
    impfset_changes=targeted_campaign,
    cost_income=CostIncome(init_cost=500_000),
)

retrofitting_and_seawall = Measure(
    name="Building Retrofit and seawall construction",
    exposures_changes=building_retrofit,
    hazard_changes=sea_wall_protection,
    cost_income=CostIncome(init_cost=1_000_000),
)

meas_set = MeasureSet([campaign, retrofitting_and_seawall])

In [12]:
measure_kwargs = {
    "Targeted awareness campaign": MeasureKwargs(
        impfset={"impf_id": 1, "rescale": 0.9}
    ),
    "Building Retrofit and seawall construction": MeasureKwargs(
        exposures={"reduction_factor": 0.8}, hazard={"reduction": 0.2}
    ),
}

In [13]:
# Create the trajectory
appraiser = StaticAppraiser(
    [snap1, snap2], measure_set=meas_set, measure_kwargs=measure_kwargs
)

# Observe the results
appraiser.per_date_risk_metrics()

No group id defined in the Exposures object. Per group aai will be empty.
No group id defined in the Exposures object. Per group aai will be empty.
No group id defined in the Exposures object. Per group aai will be empty.


,date,group,measure,metric,unit,risk,reference risk,averted risk,measure implementation cost
0,2018-01-01,All,no_measure,aai,USD,2.888955e+08,2.888955e+08,0.000000e+00,0.0
1,2018-01-01,All,Targeted awareness campaign,aai,USD,2.600059e+08,2.888955e+08,2.888955e+07,-500000.0
2,2018-01-01,All,Building Retrofit and seawall construction,aai,USD,5.567388e+07,2.888955e+08,2.332216e+08,-1000000.0
3,2050-01-01,All,no_measure,aai,USD,9.252758e+08,9.252758e+08,0.000000e+00,0.0
4,2050-01-01,All,Targeted awareness campaign,aai,USD,8.327482e+08,9.252758e+08,9.252758e+07,-500000.0
5,2050-01-01,All,Building Retrofit and seawall construction,aai,USD,1.783449e+08,9.252758e+08,7.469308e+08,-1000000.0
6,2018-01-01,All,no_measure,rp_100,USD,1.365470e+10,1.365470e+10,0.000000e+00,0.0
7,2018-01-01,All,Targeted awareness campaign,rp_100,USD,1.228923e+10,1.365470e+10,1.365470e+09,-500000.0
8,2018-01-01,All,Building Retrofit and seawall construction,rp_100,USD,2.647478e+09,1.365470e+10,1.100722e+10,-1000000.0
9,2050-01-01,All,no_measure,rp_100,USD,4.096960e+10,4.096960e+10,0.000000e+00,0.0
